# Stage 3: baselines (no training)
**Use a GPU:** Runtime > Change runtime type > **T4 GPU** before running.

Run the cells in order (or Runtime > Run all). Nothing to upload.
Scores majority vote, marker-sync majority vote and the pretrained DNAFormer on all
1,000 test clusters per deletion level, payload-only.

### Cell 1: mount Drive, set paths, check data, checkpoint and GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/fyp_dna'
CODE    = f'{PROJECT}/code'
REPO    = '/content/Deep-DNA-based-storage'
DATA    = f'{PROJECT}/data/stage1_pilot'
CKPT    = f'{PROJECT}/checkpoints/DNAFormer_siamese.pth'
OUT     = f'{PROJECT}/results/stage3'
os.makedirs(CODE, exist_ok=True); os.makedirs(OUT, exist_ok=True); os.makedirs(f'{PROJECT}/logs', exist_ok=True)
print('Stage 1 data on Drive:', os.path.isfile(f'{DATA}/config.json'))
print('Checkpoint on Drive:  ', os.path.isfile(CKPT), '(made by the Stage 2 notebook)')
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: install the small package DNAFormer needs

In [ ]:
!pip install -q einops

### Cell 3: write the Stage 1 helpers (marker layout, marker removal)

In [ ]:
%%writefile /content/drive/MyDrive/fyp_dna/code/stage1_generate.py
"""
Stage 1: paired dataset generator for the marker vs. marker-free DNAFormer study.

Makes three paired "arms" from the SAME random payloads and the SAME noise seeds:
  marker_free      128 payload bases, no markers
  marker_periodic  (24 payload + 8 marker) x 4   = 96 payload + 32 marker
  marker_endblock  96 payload, then the 4 markers together at the end (control)

Noisy reads come from DNAFormer's own simulator
(Deep-DNA-based-storage/DataGenerator/cluster_generator.py). We only give it our
own error rates; we do not edit the repository.

Usage (Colab or local):
  python stage1_generate.py --repo /content/Deep-DNA-based-storage --out /content/drive/MyDrive/fyp_dna/data/stage1_pilot
  python stage1_generate.py --repo ... --out ... --tiny      # quick smoke test
"""
import argparse
import csv
import hashlib
import json
import os
import random
import subprocess
import sys
import time

import numpy as np

# ---------------------------------------------------------------------------
# Fixed experiment settings (saved to config.json)
# ---------------------------------------------------------------------------
BASES = "ACGT"
STRAND_LEN = 128
MARKER = "ACGTCAGT"
MARKER_INTERVAL = 24          # payload bases between markers
N_MARKERS = 4
N_READS = 10
DELETION_LEVELS = [0.01, 0.03, 0.05, 0.08]
INSERTION_RATE = 0.01
SUBSTITUTION_RATE = 0.01
LONG_DELETION_RATE = 0.0      # all deletions are single-base; change later if wanted

ARMS = ["marker_free", "marker_periodic", "marker_endblock"]

# Seed ranges never overlap between splits, so the same payload cannot appear in two splits.
SPLIT_SEED_BASE = {"train": 10_000_000, "val": 20_000_000, "test": 30_000_000}
LEVEL_SEED_STRIDE = 1_000_000  # each deletion level gets its own block of seeds
NOISE_SEED_OFFSET = 1_000_000_000
SHUFFLE_SEED = 12345

PILOT_SIZES = {"train": 2500, "val": 250, "test": 1000}   # clusters PER deletion level
TINY_SIZES = {"train": 20, "val": 5, "test": 10}


def level_tag(d):
    return "del%02d" % round(d * 100)


# ---------------------------------------------------------------------------
# Strand construction
# ---------------------------------------------------------------------------
def make_payload128(payload_seed):
    rng = np.random.default_rng(payload_seed)
    return "".join(BASES[i] for i in rng.integers(0, 4, STRAND_LEN))


def build_strand(arm, payload128):
    """Return (clean_strand, payload, marker_positions)."""
    if arm == "marker_free":
        return payload128, payload128, []
    payload96 = payload128[: MARKER_INTERVAL * N_MARKERS]  # same seed -> first 96 bases shared
    if arm == "marker_periodic":
        parts, positions = [], []
        for k in range(N_MARKERS):
            parts.append(payload96[k * MARKER_INTERVAL:(k + 1) * MARKER_INTERVAL])
            start = sum(len(p) for p in parts)
            parts.append(MARKER)
            positions.append([start, start + len(MARKER)])
        return "".join(parts), payload96, positions
    if arm == "marker_endblock":
        block = MARKER * N_MARKERS
        start = len(payload96)
        positions = [[start + k * len(MARKER), start + (k + 1) * len(MARKER)] for k in range(N_MARKERS)]
        return payload96 + block, payload96, positions
    raise ValueError(arm)


def remove_markers(arm, strand):
    """Payload-only view of a 128-base strand (used by later stages for scoring)."""
    if arm == "marker_free":
        return strand
    if arm == "marker_periodic":
        step = MARKER_INTERVAL + len(MARKER)
        return "".join(strand[k * step:k * step + MARKER_INTERVAL] for k in range(N_MARKERS))
    if arm == "marker_endblock":
        return strand[: MARKER_INTERVAL * N_MARKERS]
    raise ValueError(arm)


def payload_to_bytes(payload):
    """2 bits per base (A=00 C=01 G=10 T=11). 128 bases -> 32 bytes, 96 bases -> 24 bytes."""
    bits = "".join(format(BASES.index(b), "02b") for b in payload)
    return bytes(int(bits[i:i + 8], 2) for i in range(0, len(bits), 8))


# ---------------------------------------------------------------------------
# DNAFormer simulator wrapper
# ---------------------------------------------------------------------------
def load_simulator(repo):
    gen_dir = os.path.join(repo, "DataGenerator")
    if not os.path.isfile(os.path.join(gen_dir, "cluster_generator.py")):
        sys.exit("ERROR: cluster_generator.py not found in %s. Check --repo." % gen_dir)
    sys.path.insert(0, gen_dir)
    from cluster_generator import ClusterGenerator  # noqa: E402
    return ClusterGenerator


def error_rates(deletion_rate):
    """Rates in the simulator's own format.

    The simulator gives each base one error chance equal to the SUM of the general
    rates, then picks the error type with the per-base weights. Using the same numbers
    for both makes the per-base probabilities exactly sub/ins/del.
    """
    general = {"s": SUBSTITUTION_RATE, "i": INSERTION_RATE, "d": deletion_rate, "ld": LONG_DELETION_RATE}
    per_base = {b: {"s": SUBSTITUTION_RATE, "i": 0.25, "pi": INSERTION_RATE,
                    "d": deletion_rate, "ld": LONG_DELETION_RATE} for b in BASES}
    # 'i' only chooses WHICH base is inserted (uniform here); 'pi' is the insertion rate.
    return general, per_base


def simulate_reads(ClusterGenerator, clean, deletion_rate, noise_seed):
    random.seed(noise_seed)          # the simulator uses Python's global random module
    np.random.seed(noise_seed % (2 ** 32))
    general, per_base = error_rates(deletion_rate)
    gen = ClusterGenerator(general, per_base, clean, min_copies=N_READS, max_copies=N_READS)
    gen.cluster_size = N_READS       # the repo's randint(min, max+1) could give 11; force exactly 10
    gen.generate_cluster(delta=0)    # delta=0: no random jitter of the rates
    assert len(gen.copies) == N_READS
    return list(gen.copies)


# ---------------------------------------------------------------------------
# Cluster plan
# ---------------------------------------------------------------------------
def plan_clusters(sizes):
    """List of dicts: split, subset, deletion_rate, payload_seed, noise_seed, pair_id."""
    plan = []
    for split in ["train", "val", "test"]:
        for li, d in enumerate(DELETION_LEVELS):
            for i in range(sizes[split]):
                seed = SPLIT_SEED_BASE[split] + li * LEVEL_SEED_STRIDE + i
                subset = split if split != "test" else "test_" + level_tag(d)
                plan.append({
                    "pair_id": "%s_%s_%06d" % (split, level_tag(d), i),
                    "split": split,
                    "subset": subset,
                    "deletion_rate": d,
                    "payload_seed": seed,
                    "noise_seed": seed + NOISE_SEED_OFFSET,
                })
    # Mix the deletion levels inside train and val (fixed shuffle seed).
    rng = random.Random(SHUFFLE_SEED)
    for split in ["train", "val"]:
        idx = [k for k, c in enumerate(plan) if c["split"] == split]
        items = [plan[k] for k in idx]
        rng.shuffle(items)
        for k, c in zip(idx, items):
            plan[k] = c
    return plan


def make_cluster(ClusterGenerator, arm, c):
    payload128 = make_payload128(c["payload_seed"])
    clean, payload, marker_pos = build_strand(arm, payload128)
    reads = simulate_reads(ClusterGenerator, clean, c["deletion_rate"], c["noise_seed"])
    n_marker = len(clean) - len(payload)
    return {
        "cluster_id": "%s__%s" % (arm, c["pair_id"]),
        "pair_id": c["pair_id"],
        "arm": arm,
        "split": c["split"],
        "subset": c["subset"],
        "seed": c["payload_seed"],
        "payload_seed": c["payload_seed"],
        "noise_seed": c["noise_seed"],
        "marker_present": arm != "marker_free",
        "marker_layout": {"marker_free": "none", "marker_periodic": "periodic",
                          "marker_endblock": "end_block"}[arm],
        "marker_sequence": MARKER if arm != "marker_free" else "",
        "marker_interval": MARKER_INTERVAL if arm == "marker_periodic" else None,
        "marker_positions": marker_pos,
        "marker_overhead": n_marker / len(clean),
        "payload_length": len(payload),
        "strand_length": len(clean),
        "clean_strand": clean,
        "payload_strand": payload,
        "payload_bytes_hex": payload_to_bytes(payload).hex(),
        "noisy_reads": reads,
        "read_lengths": [len(r) for r in reads],
        "deletion_rate": c["deletion_rate"],
        "insertion_rate": INSERTION_RATE,
        "substitution_rate": SUBSTITUTION_RATE,
        "long_deletion_rate": LONG_DELETION_RATE,
    }


# ---------------------------------------------------------------------------
# Writing
# ---------------------------------------------------------------------------
def write_subset(folder, clusters):
    os.makedirs(folder, exist_ok=True)
    with open(os.path.join(folder, "clean.fasta"), "w", newline="\n") as f:
        for c in clusters:
            f.write(">%s\n%s\n" % (c["cluster_id"], c["clean_strand"]))
    with open(os.path.join(folder, "noisy_reads.fasta"), "w", newline="\n") as f:
        for c in clusters:
            for j, r in enumerate(c["noisy_reads"]):
                f.write(">%s|read_%d\n%s\n" % (c["cluster_id"], j, r))
    with open(os.path.join(folder, "metadata.json"), "w", newline="\n") as f:
        json.dump(clusters, f, separators=(",", ":"))


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def repo_commit(repo):
    try:
        return subprocess.check_output(["git", "-C", repo, "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return "unknown"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True, help="path to Deep-DNA-based-storage")
    ap.add_argument("--out", required=True, help="output folder")
    ap.add_argument("--tiny", action="store_true", help="very small smoke-test dataset")
    args = ap.parse_args()

    sizes = TINY_SIZES if args.tiny else PILOT_SIZES
    ClusterGenerator = load_simulator(args.repo)
    plan = plan_clusters(sizes)
    os.makedirs(args.out, exist_ok=True)

    t0 = time.time()
    manifest_rows = []
    for arm in ARMS:
        by_subset = {}
        for k, c in enumerate(plan):
            cl = make_cluster(ClusterGenerator, arm, c)
            by_subset.setdefault(c["subset"], []).append(cl)
            if (k + 1) % 2000 == 0:
                print("  %s: %d/%d clusters (%.0fs)" % (arm, k + 1, len(plan), time.time() - t0), flush=True)
        for subset, clusters in by_subset.items():
            folder = os.path.join(args.out, arm, subset)
            write_subset(folder, clusters)
            for cl in clusters:
                manifest_rows.append({
                    "cluster_id": cl["cluster_id"], "pair_id": cl["pair_id"], "arm": arm,
                    "split": cl["split"], "subset": subset,
                    "deletion_rate": cl["deletion_rate"], "insertion_rate": cl["insertion_rate"],
                    "substitution_rate": cl["substitution_rate"],
                    "payload_seed": cl["payload_seed"], "noise_seed": cl["noise_seed"],
                    "marker_present": cl["marker_present"], "marker_layout": cl["marker_layout"],
                    "marker_sequence": cl["marker_sequence"], "marker_interval": cl["marker_interval"],
                    "payload_length": cl["payload_length"], "strand_length": cl["strand_length"],
                    "marker_overhead": cl["marker_overhead"], "n_reads": len(cl["noisy_reads"]),
                    "mean_read_length": round(float(np.mean(cl["read_lengths"])), 2),
                    "folder": os.path.join(arm, subset).replace("\\", "/"),
                })
        print("%s done (%.0fs)" % (arm, time.time() - t0), flush=True)

    with open(os.path.join(args.out, "dataset_manifest.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(manifest_rows[0].keys()))
        w.writeheader()
        w.writerows(manifest_rows)

    config = {
        "stage": 1, "mode": "tiny" if args.tiny else "pilot",
        "clusters_per_level": sizes, "deletion_levels": DELETION_LEVELS,
        "insertion_rate": INSERTION_RATE, "substitution_rate": SUBSTITUTION_RATE,
        "long_deletion_rate": LONG_DELETION_RATE, "strand_length": STRAND_LEN,
        "marker": MARKER, "marker_interval": MARKER_INTERVAL, "n_markers": N_MARKERS,
        "n_reads": N_READS, "arms": ARMS, "split_seed_base": SPLIT_SEED_BASE,
        "level_seed_stride": LEVEL_SEED_STRIDE, "noise_seed_offset": NOISE_SEED_OFFSET,
        "shuffle_seed": SHUFFLE_SEED, "simulator": "DataGenerator/cluster_generator.py",
        "simulator_repo_commit": repo_commit(args.repo),
        "python": sys.version.split()[0], "numpy": np.__version__,
    }
    with open(os.path.join(args.out, "config.json"), "w", newline="\n") as f:
        json.dump(config, f, indent=2)

    checksums = {}
    for root, _, files in os.walk(args.out):
        for name in files:
            if name in ("checksums.json",):
                continue
            p = os.path.join(root, name)
            checksums[os.path.relpath(p, args.out).replace("\\", "/")] = sha256(p)
    with open(os.path.join(args.out, "checksums.json"), "w", newline="\n") as f:
        json.dump(dict(sorted(checksums.items())), f, indent=1)

    print("\nDONE: %d clusters per arm x %d arms in %.0fs -> %s"
          % (len(plan), len(ARMS), time.time() - t0, args.out))


if __name__ == "__main__":
    main()

### Cell 4: write the DNAFormer adapter (same as Stage 2)

In [ ]:
%%writefile /content/drive/MyDrive/fyp_dna/code/dnaf_adapter.py
"""
Minimal adapter between our Stage 1 data and the UNCHANGED DNAFormer model.

What it does:
  - builds DNAFormer exactly as the repo defines it (DNN/model_DNAFormer_siamese.py)
  - loads the pretrained checkpoint and REFUSES to continue if any weight is missing
    (the repo uses strict=False, which silently leaves layers random)
  - turns a cluster of reads into the model's input tensors, without the repo
    loader's problems (crash on Python >= 3.11, reads of length 132 turned into
    zeros, deletion-heavy reads thrown away, clusters silently swapped)

The model architecture is not touched. Input/target lengths stay 132 / 128.
"""
import os
import sys

import numpy as np

BASES = "ACGT"
BASE_IDX = {b: i for i, b in enumerate(BASES)}   # same mapping as the repo: A=0 C=1 G=2 T=3

LABEL_LEN = 128        # repo: label_length 140 - index_length 12
INPUT_LEN = 132        # repo: noisy_copies_length = 140 + 4 - 12
MAX_READS = 16         # repo: max_number_per_cluster
ORIG_MAX_DEVIATION = 4  # repo: corrupt_max_deviation

# Old checkpoint name -> current model name (found in Stage 2: same shape, renamed in code)
CHECKPOINT_RENAMES = {
    "fusion.pred_fusion": "fusion.pred_fusion_left",
    "fusion.pred_fusion_flip": "fusion.pred_fusion_right",
}


def reset_fusion_weights(model):
    """Set both fusion vectors to 1.0, i.e. what the repo's own loader ends up using."""
    import torch
    with torch.no_grad():
        model.fusion.pred_fusion_left.fill_(1.0)
        model.fusion.pred_fusion_right.fill_(1.0)


class DNAFormerConfig:
    """Same values as DNN/inference.py in the repo."""
    index_length = 12
    label_length = 140
    filter_index = True
    corrupt_max_deviation = 4
    noisy_copies_length = INPUT_LEN
    max_number_per_cluster = MAX_READS
    read_padding = "end"
    model_config = "siamese"
    n_head = 32
    activation = "gelu"
    num_layers = 12
    d_model = 1024
    alignment_filters = 128
    alignement_filters = 128   # the repo's train.py uses this misspelling
    dim_feedforward = 2048
    output_ch = 4
    enc_filters = 4
    p_dropout = 0
    class_token = 0
    use_input_scaling = False
    device = "cpu"


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
def build_model(repo, device="cpu"):
    sys.path.insert(0, os.path.join(repo, "DNN"))
    from model_DNAFormer_siamese import net  # noqa: E402
    cfg = DNAFormerConfig()
    cfg.device = device
    return net(cfg).to(device), cfg


def load_checkpoint(model, path, device="cpu"):
    """Load weights strictly. Returns a report dict; raises if anything does not match."""
    import torch
    try:
        ckpt = torch.load(path, map_location=device, weights_only=True)
        loader = "weights_only=True"
    except Exception:
        # Official checkpoint from the DNAFormer authors; may contain non-tensor objects.
        ckpt = torch.load(path, map_location=device, weights_only=False)
        loader = "weights_only=False"
    state = ckpt.get("model_state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    state = {k[7:] if k.startswith("module.") else k: v for k, v in state.items()}
    # The released checkpoint uses older names for the two fusion weight vectors.
    # The repo's strict=False loading silently skips them (they stay at 1.0).
    # We map them explicitly; the shape check below still guards the mapping.
    renamed = {}
    for old, new in CHECKPOINT_RENAMES.items():
        if old in state and new not in state:
            state[new] = state.pop(old)
            renamed[old] = new
    model_keys = set(model.state_dict().keys())
    ckpt_keys = set(state.keys())
    missing = sorted(model_keys - ckpt_keys)
    unexpected = sorted(ckpt_keys - model_keys)
    shape_bad = sorted(k for k in model_keys & ckpt_keys
                       if tuple(model.state_dict()[k].shape) != tuple(state[k].shape))
    report = {
        "loader": loader,
        "checkpoint_top_keys": sorted(ckpt.keys()) if isinstance(ckpt, dict) else [],
        "checkpoint_epoch": ckpt.get("epoch") if isinstance(ckpt, dict) else None,
        "n_tensors": len(ckpt_keys), "missing": missing, "unexpected": unexpected,
        "shape_mismatch": shape_bad, "renamed": renamed,
        "renamed_shapes": {new: tuple(state[new].shape) for new in renamed.values()},
        "renamed_stats": {new: [round(float(state[new].min()), 4), round(float(state[new].mean()), 4),
                                round(float(state[new].max()), 4)] for new in renamed.values()},
    }
    if missing or unexpected or shape_bad:
        raise RuntimeError("Checkpoint does not match the model: %s" % report)
    model.load_state_dict(state, strict=True)
    return report


# ---------------------------------------------------------------------------
# Input encoding
# ---------------------------------------------------------------------------
def select_reads(reads, policy="relaxed"):
    """Which reads go into the model.

    relaxed  (ours): keep every read; reads longer than 132 are cut to 132.
    original (repo): keep reads with |len-128| <= 4; a read of exactly 132 becomes
                     an all-zero row (repo bug). If no read survives, the repo would
                     replace the whole cluster with a random other one.
    """
    if policy == "relaxed":
        return [r[:INPUT_LEN] for r in reads[:MAX_READS]]
    if policy == "original":
        kept = [r for r in reads if abs(len(r) - LABEL_LEN) <= ORIG_MAX_DEVIATION]
        return [r if len(r) < INPUT_LEN else "" for r in kept[:MAX_READS]]
    raise ValueError(policy)


def one_hot(read, length=INPUT_LEN):
    x = np.zeros((4, length), dtype=np.float32)
    for j, b in enumerate(read[:length]):
        x[BASE_IDX[b], j] = 1.0
    return x


def encode_cluster(reads, policy="relaxed"):
    """Return (left, right, n_used) as numpy arrays of shape [16, 4, 132].

    left  = each read one-hot, zero-padded at the end (repo read_padding='end')
    right = each read reversed, then zero-padded at the end (repo siamese branch)
    Unused read slots stay all-zero, as in the repo.
    """
    used = select_reads(reads, policy)
    left = np.zeros((MAX_READS, 4, INPUT_LEN), dtype=np.float32)
    right = np.zeros((MAX_READS, 4, INPUT_LEN), dtype=np.float32)
    for i, r in enumerate(used):
        left[i] = one_hot(r)
        right[i] = one_hot(r[::-1])
    return left, right, len(used)


def make_batch(clusters, policy="relaxed", device="cpu"):
    """clusters: list of Stage 1 metadata dicts. Returns (model_input, labels).

    model_input has shape [2B, 16, 4, 132]: left branch first, right branch second,
    exactly as the repo concatenates them. labels has shape [B, 128] (class indices).
    """
    import torch
    L, R, Y = [], [], []
    for c in clusters:
        left, right, _ = encode_cluster(c["noisy_reads"], policy)
        L.append(left)
        R.append(right)
        Y.append([BASE_IDX[b] for b in c["clean_strand"]])
    x = torch.from_numpy(np.concatenate([np.stack(L), np.stack(R)], axis=0)).to(device)
    y = torch.tensor(Y, dtype=torch.long, device=device)
    return x, y


def decode(pred_logits):
    """[B, 4, 128] logits -> list of DNA strings."""
    idx = pred_logits.argmax(dim=1).cpu().numpy()
    return ["".join(BASES[i] for i in row) for row in idx]


# ---------------------------------------------------------------------------
# Scoring helpers
# ---------------------------------------------------------------------------
def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

### Cell 5: write the Stage 3 script

In [ ]:
%%writefile /content/drive/MyDrive/fyp_dna/code/stage3_baselines.py
"""
Stage 3: baseline evaluation on all test clusters. No training.

Methods
  mv            plain majority vote: line the reads up from the left, vote per position.
                Run on all 3 arms.
  mv_sync       marker-sync majority vote (periodic arm only): find the 4 markers in each
                read, cut the read into the 4 payload segments between them, then vote
                inside each segment. Each marker lets the vote re-align after deletions.
  dnaformer     pretrained DNAFormer, no fine-tuning, relaxed read policy (all reads kept).
                Run on all 3 arms.

Scoring is payload-only: marker bases are removed from the prediction before comparing it
with the true payload (128 bases for marker_free, 96 for the marker arms).

Metrics per (method, arm, deletion level)
  exact recovery rate with 95% Wilson CI
  normalized edit distance (edit distance / payload length), mean with 95% CI
  per-base accuracy (same base at the same payload position), mean with 95% CI
  file-level recovery: 25 consecutive test strands = 1 "file"; a file counts as
    recovered only if all 25 strands are exact (no outer error-correcting code), Wilson CI
  runtime (s total, ms per cluster) and peak GPU memory (DNAFormer only)

Outputs (in --out)
  predictions.csv   one row per (method, arm, cluster)
  results.csv       one row per (method, arm, deletion level)
  stage3_log.json   environment, checkpoint report, settings

Usage
  python stage3_baselines.py --repo REPO --data DATA --ckpt CKPT --out OUT
  python stage3_baselines.py --repo REPO --data DATA --out OUT --no-torch    # majority votes only
"""
import argparse
import csv
import json
import math
import os
import platform
import sys
import time

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import dnaf_adapter as A  # noqa: E402
import stage1_generate as G  # noqa: E402

FILE_SIZE = 25          # strands per "file" for file-level recovery
MARKER_MAX_ED = 2       # a marker match may have at most this many edits
MARKER_SEARCH = 6       # search +/- this many bases around where the marker should be
DNAF_BATCH = 32
Z95 = 1.959964


# ---------------------------------------------------------------------------
# Majority vote
# ---------------------------------------------------------------------------
def vote(columns_reads, length):
    """Left-aligned positional majority vote over a list of strings.

    Position j is voted on by every string that is long enough. Ties go to the first base
    in A, C, G, T order. Positions no string reaches get 'A' (counted as a guess)."""
    out = []
    for j in range(length):
        counts = [0, 0, 0, 0]
        for r in columns_reads:
            if j < len(r):
                counts[A.BASE_IDX[r[j]]] += 1
        out.append(A.BASES[int(np.argmax(counts))])
    return "".join(out)


def mv_plain(reads):
    return vote(reads, G.STRAND_LEN)


def find_marker(read, expected_start):
    """Best approximate match of the marker near expected_start.

    Semi-global alignment: the marker must match completely, the read may start and end
    anywhere inside the search window. Returns (start, end, edits) or None if the best
    match has more than MARKER_MAX_ED edits."""
    m = G.MARKER
    lo = max(0, expected_start - MARKER_SEARCH)
    hi = min(len(read), expected_start + len(m) + MARKER_SEARCH)
    text = read[lo:hi]
    if not text:
        return None
    exact = text.find(m)
    if exact >= 0:
        # take the exact hit closest to the expected position
        best, s = None, exact
        while s >= 0:
            if best is None or abs(lo + s - expected_start) < abs(lo + best - expected_start):
                best = s
            s = text.find(m, s + 1)
        return lo + best, lo + best + len(m), 0
    # D[i][j]: edits to align marker[:i] with a substring of text ending at j
    n = len(text)
    prev = [0] * (n + 1)              # row i=0: free start anywhere
    prev_start = list(range(n + 1))   # where that alignment starts in text
    for i in range(1, len(m) + 1):
        cur = [i] + [0] * n
        cur_start = [0] + [0] * n
        for j in range(1, n + 1):
            sub = prev[j - 1] + (m[i - 1] != text[j - 1])
            dele = prev[j] + 1        # marker base missing from read
            ins = cur[j - 1] + 1      # extra base in read
            best = min(sub, dele, ins)
            cur[j] = best
            if best == sub:
                cur_start[j] = prev_start[j - 1]
            elif best == dele:
                cur_start[j] = prev_start[j]
            else:
                cur_start[j] = cur_start[j - 1]
        prev, prev_start = cur, cur_start
    best_j = min(range(1, n + 1), key=lambda j: (prev[j], abs(lo + j - len(m) - expected_start)))
    if prev[best_j] > MARKER_MAX_ED:
        return None
    return lo + prev_start[best_j], lo + best_j, prev[best_j]


def split_at_markers(read):
    """Cut one periodic-arm read into its 4 payload segments.

    Markers are searched one after the other; the expected position of the next marker is
    24 bases after the end of the previous one, so a deletion earlier in the read does not
    throw off the later searches. If a marker is not found, we assume it sits where expected.
    Returns (segments, n_markers_found)."""
    segments, found, pos = [], 0, 0
    for _ in range(G.N_MARKERS):
        expected = pos + G.MARKER_INTERVAL
        hit = find_marker(read, expected)
        if hit is None:
            start, end = min(expected, len(read)), min(expected + len(G.MARKER), len(read))
        else:
            start, end, _ = hit
            found += 1
        segments.append(read[pos:start])
        pos = end
    return segments, found


def mv_sync(reads):
    """Marker-sync majority vote. Returns (payload96, fraction of markers found)."""
    per_read = [split_at_markers(r) for r in reads]
    payload = "".join(vote([segs[k] for segs, _ in per_read], G.MARKER_INTERVAL)
                      for k in range(G.N_MARKERS))
    found = sum(f for _, f in per_read) / (G.N_MARKERS * len(reads))
    return payload, found


# ---------------------------------------------------------------------------
# Scoring and statistics
# ---------------------------------------------------------------------------
def score(pred_payload, true_payload):
    ed = A.edit_distance(pred_payload, true_payload)
    same = sum(a == b for a, b in zip(pred_payload, true_payload))
    n = len(true_payload)
    return {"edit_distance": ed, "ned": ed / n, "base_acc": same / n, "exact": int(pred_payload == true_payload)}


def wilson(k, n):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    den = 1 + Z95 ** 2 / n
    mid = (p + Z95 ** 2 / (2 * n)) / den
    half = Z95 * math.sqrt(p * (1 - p) / n + Z95 ** 2 / (4 * n * n)) / den
    return max(0.0, mid - half), min(1.0, mid + half)


def mean_ci(x):
    x = np.asarray(x, dtype=float)
    m = float(x.mean())
    half = Z95 * float(x.std(ddof=1)) / math.sqrt(len(x)) if len(x) > 1 else float("nan")
    return m, m - half, m + half


def summarize(method, arm, level, rows, runtime_s, gpu_peak_gb):
    exact = [r["exact"] for r in rows]
    k, n = sum(exact), len(exact)
    lo, hi = wilson(k, n)
    ned_m, ned_lo, ned_hi = mean_ci([r["ned"] for r in rows])
    acc_m, acc_lo, acc_hi = mean_ci([r["base_acc"] for r in rows])
    files = {}
    for r in rows:
        files.setdefault(r["file_id"], []).append(r["exact"])
    full = [f for f in files.values() if len(f) == FILE_SIZE]
    fk = sum(all(f) for f in full)
    flo, fhi = wilson(fk, len(full))
    payload_len = rows[0]["payload_length"]
    return {
        "method": method, "arm": arm, "deletion_rate": level, "n_clusters": n,
        "payload_length": payload_len, "info_rate": payload_len / G.STRAND_LEN,
        "exact_recovery": k / n, "exact_ci_low": lo, "exact_ci_high": hi,
        "ned_mean": ned_m, "ned_ci_low": ned_lo, "ned_ci_high": ned_hi,
        "base_acc_mean": acc_m, "base_acc_ci_low": acc_lo, "base_acc_ci_high": acc_hi,
        "n_files": len(full), "files_recovered": fk,
        "file_recovery": fk / len(full) if full else float("nan"),
        "file_ci_low": flo, "file_ci_high": fhi,
        "runtime_s": runtime_s, "ms_per_cluster": 1000 * runtime_s / n,
        "gpu_peak_gb": gpu_peak_gb,
        "markers_found": (float(np.mean([r["markers_found"] for r in rows]))
                          if rows[0]["markers_found"] != "" else ""),
    }


# ---------------------------------------------------------------------------
# Runner
# ---------------------------------------------------------------------------
PRED_FIELDS = ["method", "arm", "deletion_rate", "cluster_id", "pair_id", "file_id", "payload_length",
               "true_payload", "pred_payload", "edit_distance", "ned", "base_acc", "exact",
               "markers_found"]


def load_test(data, arm, level, limit):
    with open(os.path.join(data, arm, "test_" + G.level_tag(level), "metadata.json")) as f:
        clusters = json.load(f)
    return clusters[:limit] if limit else clusters


def make_rows(method, arm, level, clusters, preds, markers_found=None):
    rows = []
    for i, (c, p) in enumerate(zip(clusters, preds)):
        s = score(p, c["payload_strand"])
        rows.append({
            "method": method, "arm": arm, "deletion_rate": level,
            "cluster_id": c["cluster_id"], "pair_id": c["pair_id"],
            "file_id": "%s_file%03d" % (G.level_tag(level), i // FILE_SIZE),
            "payload_length": len(c["payload_strand"]),
            "true_payload": c["payload_strand"], "pred_payload": p,
            "markers_found": "" if markers_found is None else round(markers_found[i], 3), **s,
        })
    return rows


def run_mv(method, arm, clusters):
    t0 = time.time()
    if method == "mv":
        preds, found = [G.remove_markers(arm, mv_plain(c["noisy_reads"])) for c in clusters], None
    else:
        out = [mv_sync(c["noisy_reads"]) for c in clusters]
        preds, found = [p for p, _ in out], [f for _, f in out]
    return preds, found, time.time() - t0


def run_dnaformer(model, arm, clusters, device):
    import torch
    if device == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    preds = []
    with torch.inference_mode():
        for i in range(0, len(clusters), DNAF_BATCH):
            chunk = clusters[i:i + DNAF_BATCH]
            x, _ = A.make_batch(chunk, "relaxed", device)
            logits = model(x)["pred"]
            if not torch.isfinite(logits).all():
                raise RuntimeError("DNAFormer produced NaN/inf outputs")
            preds += [G.remove_markers(arm, p) for p in A.decode(logits)]
    if device == "cuda":
        torch.cuda.synchronize()
    runtime = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else ""
    return preds, runtime, peak


def write_csv(path, rows, fields):
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for r in rows:
            w.writerow({k: (round(v, 6) if isinstance(v, float) else v) for k, v in r.items()})


def print_table(results):
    print("\n%-10s %-16s %5s  %-22s %-8s %-8s %-6s %s" % ("method", "arm", "del", "exact recovery [95% CI]",
                                                      "NED", "base acc", "files", "ms/cl"))
    for r in results:
        print("%-10s %-16s %4.0f%%  %5.1f%% [%5.1f, %5.1f]    %.4f   %.4f   %2d/%-3d %.1f" % (
            r["method"], r["arm"], 100 * r["deletion_rate"], 100 * r["exact_recovery"],
            100 * r["exact_ci_low"], 100 * r["exact_ci_high"], r["ned_mean"], r["base_acc_mean"],
            r["files_recovered"], r["n_files"], r["ms_per_cluster"]))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--ckpt", default=None)
    ap.add_argument("--out", required=True)
    ap.add_argument("--limit", type=int, default=0, help="clusters per level (0 = all)")
    ap.add_argument("--no-torch", action="store_true", help="majority votes only")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    levels = G.DELETION_LEVELS
    log = {"python": platform.python_version(), "file_size": FILE_SIZE, "marker_max_ed": MARKER_MAX_ED,
           "marker_search": MARKER_SEARCH, "read_policy": "relaxed", "limit": args.limit}

    model, device = None, "cpu"
    if not args.no_torch:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        log["torch"] = torch.__version__
        log["device"] = device
        if device == "cuda":
            log["gpu"] = torch.cuda.get_device_name(0)
        else:
            print("WARNING: no GPU, DNAFormer will be slow. Runtime > Change runtime type > T4 GPU.")
        if not args.ckpt or not os.path.isfile(args.ckpt):
            sys.exit("ERROR: checkpoint not found: %s" % args.ckpt)
        model, _ = A.build_model(args.repo, device)
        rep = A.load_checkpoint(model, args.ckpt, device)
        model.eval()
        print("Checkpoint loaded strictly: %d tensors, renamed %s" % (rep["n_tensors"], rep["renamed"]))
        log["checkpoint"] = {k: rep[k] for k in ("loader", "n_tensors", "checkpoint_epoch", "renamed")}

    runs = [("mv", arm) for arm in G.ARMS] + [("mv_sync", "marker_periodic")]
    if model is not None:
        runs += [("dnaformer", arm) for arm in G.ARMS]

    all_rows, results = [], []
    for method, arm in runs:
        for level in levels:
            clusters = load_test(args.data, arm, level, args.limit)
            found = None
            if method == "dnaformer":
                preds, runtime, peak = run_dnaformer(model, arm, clusters, device)
            else:
                preds, found, runtime = run_mv(method, arm, clusters)
                peak = ""
            rows = make_rows(method, arm, level, clusters, preds, found)
            res = summarize(method, arm, level, rows, runtime, peak)
            all_rows += rows
            results.append(res)
            print("  done %-10s %-16s %2.0f%%: exact %5.1f%%, NED %.4f (%.1fs)" % (
                method, arm, 100 * level, 100 * res["exact_recovery"], res["ned_mean"], runtime), flush=True)

    write_csv(os.path.join(args.out, "predictions.csv"), all_rows, PRED_FIELDS)
    write_csv(os.path.join(args.out, "results.csv"), results, list(results[0].keys()))
    with open(os.path.join(args.out, "stage3_log.json"), "w") as f:
        json.dump(log, f, indent=1)

    print_table(results)
    if model is not None and device == "cuda":
        peaks = [r["gpu_peak_gb"] for r in results if r["method"] == "dnaformer"]
        print("\nDNAFormer inference peak GPU memory: %.2f GB (batch %d)" % (max(peaks), DNAF_BATCH))
    mf = [r for r in results if r["method"] == "mv_sync"]
    if mf:
        print("Markers found by mv_sync (share of 4 markers x 10 reads): " +
              ", ".join("%.0f%%: %.1f%%" % (100 * r["deletion_rate"], 100 * r["markers_found"]) for r in mf))
    n_expected = len(runs) * len(levels)
    ok = len(results) == n_expected and all(r["n_clusters"] == results[0]["n_clusters"] for r in results)
    print("\nSaved %d prediction rows and %d result rows to %s" % (len(all_rows), len(results), args.out))
    print("STAGE 3 DONE" if ok else "STAGE 3 INCOMPLETE: expected %d result rows" % n_expected)


if __name__ == "__main__":
    main()

### Cell 6: get the DNAFormer repo (same pinned commit as Stages 1 and 2)

In [ ]:
!rm -rf {REPO}
!git clone -q https://github.com/itaiorr/Deep-DNA-based-storage {REPO}
!git -C {REPO} checkout -q 167a68261719ea458e79c40fcc485ae768c1a64a
!git -C {REPO} rev-parse HEAD

### Cell 7: quick smoke test (25 clusters per level, about 1 minute)
Should end with `STAGE 3 DONE`. If it fails, paste the error to Claude.

In [ ]:
!cd {CODE} && python stage3_baselines.py --repo {REPO} --data {DATA} --ckpt {CKPT} --out /content/stage3_smoke --limit 25 2>&1 | grep -v Warning | tail -8

### Cell 8: full run (all 1,000 test clusters per level, about 5 minutes)
**Paste the whole output of this cell back to Claude.**

In [ ]:
!cd {CODE} && python stage3_baselines.py --repo {REPO} --data {DATA} --ckpt {CKPT} --out {OUT} 2>&1 | grep -v Warning | tee {PROJECT}/logs/stage3_baselines.txt
!ls -la {OUT}

---
**Results are saved to** `MyDrive/fyp_dna/results/stage3/`:
- `results.csv`: one row per method, version and deletion level (the summary)
- `predictions.csv`: one row per test strand (true payload, prediction, scores)
- `stage3_log.json`: settings and environment

**Methods:** `mv` = plain majority vote, `mv_sync` = majority vote that re-aligns at each marker
(periodic version only), `dnaformer` = pretrained DNAFormer without fine-tuning.